# Federated Evo2-1B LoRA fine-tuning with BioNeMo and NVFlare

This walkthrough prepares a public splice-site dataset, converts Evo2-1B for Megatron Bridge, exports one
common LoRA and classification-head initialization, runs sample-weighted FedAvg across three simulated
sites, and reloads the final global checkpoint for evaluation.

Only the LoRA adapters and classification head are federated. The frozen Evo2 backbone stays at each site.
Each task uses fresh local optimizer, scheduler, RNG, and Transformer Engine state. The workflow runs
sequentially on one H100.


## Setup

Build the pinned image and enter it with `./start_evo2.sh` as described in [README.md](./README.md). The
README records the PyTorch image, BioNeMo Recipes, Megatron Bridge, causal-conv1d, model, and dataset pins.


In [ ]:
import json
import os
import subprocess
from pathlib import Path

EXAMPLE_DIR = Path.cwd().resolve()
if not (EXAMPLE_DIR / "job.py").is_file():
    EXAMPLE_DIR = Path("/workspace/nvflare/examples/advanced/bionemo/evo2")
assert (EXAMPLE_DIR / "job.py").is_file(), f"Run from the Evo2 example directory: {EXAMPLE_DIR}"
os.chdir(EXAMPLE_DIR)

DATA_DIR = EXAMPLE_DIR / "data"
MODEL_DIR = EXAMPLE_DIR / "models"
RESULTS_DIR = EXAMPLE_DIR / "results" / "fedavg_lora"
BASE_CHECKPOINT = MODEL_DIR / "evo2_1b_bf16_mbridge"
INITIAL_CHECKPOINT = MODEL_DIR / "evo2_lora_init.pt"
WORKSPACE = Path("/tmp/nvflare/evo2_splice_fedavg")

NUM_CLIENTS = 3
NUM_ROUNDS = 10
LOCAL_STEPS = 20
SEQ_LENGTH = 600
MICRO_BATCH_SIZE = 4
GLOBAL_BATCH_SIZE = 32
SEED = 1234
LORA_DIM = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.1

MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print("Example:", EXAMPLE_DIR)
print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES", "<not set>"))


## 1. Prepare the public dataset

The pinned `splice_sites_all` task contains 30,000 training and 3,000 official test sequences of 600 bases.
This compact run reserves 10% of the source training split for validation and selects a stratified
3,000-sequence showcase. Preparation audits exact duplicates and overlapping genomic windows before writing
three deterministic IID site partitions. No sequence data is bundled with NVFlare.


In [ ]:
subprocess.run(
    [
        "python3", "prepare_data.py",
        "--output-dir", str(DATA_DIR),
        "--num-sites", str(NUM_CLIENTS),
        "--partition", "iid",
        "--showcase-size", "3000",
        "--validation-fraction", "0.1",
        "--seed", "42",
    ],
    check=True,
)


In [ ]:
manifest = json.loads((DATA_DIR / "manifest.json").read_text())
print(json.dumps({
    "source": manifest["source"],
    "settings": manifest["settings"],
    "counts": manifest["counts"],
    "label_histograms": manifest["label_histograms"],
    "audit": manifest["audit"],
}, indent=2))


## 2. Download and convert Evo2-1B

BioNeMo downloads `evo2/1b-8k-bf16:1.0` and converts it once from NeMo 2 to the Megatron Bridge checkpoint
format. All clients reuse these identical frozen weights and tokenizer settings.


In [ ]:
subprocess.run(
    ["python3", "prepare_base_checkpoint.py", "--output", str(BASE_CHECKPOINT)],
    check=True,
)
print("Megatron Bridge checkpoint:", BASE_CHECKPOINT)


## 3. Export a common LoRA and classification-head initialization

This step instantiates Evo2 once and saves only its rank-16 LoRA and classification-head tensors. It uses
the shared validation split to construct the zero-step data module and does not optimize on those rows. The
server uses this float32 checkpoint as the identical starting state for all sites.


In [ ]:
subprocess.run(
    [
        "torchrun", "--standalone", "--nproc_per_node=1", "prepare_initial_model.py",
        "--base-checkpoint", str(BASE_CHECKPOINT),
        "--data-file", str(DATA_DIR / "validation.jsonl"),
        "--output", str(INITIAL_CHECKPOINT),
        "--work-dir", "/tmp/nvflare/evo2_initialize",
        "--seq-length", str(SEQ_LENGTH),
        "--seed", str(SEED),
        "--lora-dim", str(LORA_DIM),
        "--lora-alpha", str(LORA_ALPHA),
        "--lora-dropout", str(LORA_DROPOUT),
    ],
    check=True,
)


## 4. Run three-site federated LoRA training

Each client trains for 20 optimizer steps per round. NVFlare exchanges only LoRA and head differences and
weights each update by the site's audited number of training examples. `job.py` uses one simulator thread,
and a workspace lock keeps its external BioNeMo trainers sequential on the GPU.

For a final-commit GPU smoke test, first set `NUM_CLIENTS = 2`, `NUM_ROUNDS = 1`, `LOCAL_STEPS = 4`, and use
a new workspace. Confirm that both trainers exit, both frozen-backbone checks pass, and the resulting global
checkpoint reloads with the evaluation cells below. Then restore the compact defaults for the full example.


In [ ]:
federated_command = [
    "python3", "job.py",
    "--data-dir", str(DATA_DIR),
    "--initial-checkpoint", str(INITIAL_CHECKPOINT),
    "--base-checkpoint", str(BASE_CHECKPOINT),
    "--workspace", str(WORKSPACE),
    "--num-clients", str(NUM_CLIENTS),
    "--num-rounds", str(NUM_ROUNDS),
    "--local-steps", str(LOCAL_STEPS),
    "--seq-length", str(SEQ_LENGTH),
    "--micro-batch-size", str(MICRO_BATCH_SIZE),
    "--global-batch-size", str(GLOBAL_BATCH_SIZE),
    "--learning-rate", "0.0005",
    "--min-learning-rate", "0.00005",
    "--warmup-iters", "2",
    "--eval-iters", "10",
    "--seed", str(SEED),
    "--lora-dim", str(LORA_DIM),
    "--lora-alpha", str(LORA_ALPHA),
    "--lora-dropout", str(LORA_DROPOUT),
    "--gpu", "[0]",
]
subprocess.run(federated_command, check=True)


## 5. Reload and evaluate the final global checkpoint

Read the exact final checkpoint path from `run_summary.json`. Evaluation rebuilds the frozen backbone,
strictly loads the saved float32 LoRA and head tensors at the BF16 model boundary, verifies the official test
file against the preparation manifest, and requires every test row exactly once.


In [ ]:
run_summary = json.loads((WORKSPACE / "run_summary.json").read_text())
GLOBAL_CHECKPOINT = Path(run_summary["global_checkpoint"])
assert GLOBAL_CHECKPOINT.is_file(), GLOBAL_CHECKPOINT
print("Final global checkpoint:", GLOBAL_CHECKPOINT)

evaluation_report = RESULTS_DIR / "evaluation.json"
confusion_matrix = RESULTS_DIR / "confusion_matrix.png"
subprocess.run(
    [
        "torchrun", "--standalone", "--nproc_per_node=1", "evaluate.py",
        "--checkpoint", str(GLOBAL_CHECKPOINT),
        "--base-checkpoint", str(BASE_CHECKPOINT),
        "--test-file", str(DATA_DIR / "test.jsonl"),
        "--manifest", str(DATA_DIR / "manifest.json"),
        "--output", str(evaluation_report),
        "--confusion-matrix", str(confusion_matrix),
        "--work-dir", "/tmp/nvflare/evo2_splice_evaluation",
        "--seq-length", str(SEQ_LENGTH),
        "--micro-batch-size", str(MICRO_BATCH_SIZE),
        "--global-batch-size", str(GLOBAL_BATCH_SIZE),
        "--seed", str(SEED),
        "--lora-dim", str(LORA_DIM),
        "--lora-alpha", str(LORA_ALPHA),
        "--lora-dropout", str(LORA_DROPOUT),
    ],
    check=True,
)


In [ ]:
from IPython.display import Image, display

metrics = json.loads(evaluation_report.read_text())
print(json.dumps({
    "accuracy": metrics["accuracy"],
    "macro_f1": metrics["macro_f1"],
    "num_examples": metrics["num_examples"],
    "runtime_seconds": metrics["runtime_seconds"],
    "peak_gpu_memory_mebibytes": metrics["peak_gpu_memory_mebibytes"],
    "checkpoint_mebibytes": metrics["checkpoint_mebibytes"],
    "confusion_matrix": metrics["confusion_matrix"],
}, indent=2))
display(Image(filename=metrics["confusion_matrix_plot"]))


## Execution and data notes

- Keep one Evo2 job on the GPU at a time. The workspace lock does not coordinate separate workspaces.
- Lower the microbatch if memory is tight while keeping the global batch divisible by it.
- A ten-round, three-site run initializes Evo2 30 times, so checkpoint loading can dominate short runs.
- Tensor and pipeline parallelism, multi-node training, long-context tuning, and full-backbone federation are
  outside this example.

Before downloading or using the data, review the pinned dataset card, Nucleotide Transformer license, and
GENCODE terms linked from the README. Evo2 weights and BioNeMo software have separate terms.
